# Stage 1 — GPT-2 + IOI baseline

**Capstone: Evaluating the Faithfulness of Attribution-Based Interpretability Tooling in GPT-2**

This notebook is the foundation for everything that follows. It does three things:
1. Loads GPT-2 small.
2. Builds Indirect Object Identification (IOI) sentences.
3. Measures whether the model actually prefers the correct name, using the **logit difference** metric.

**How to run:** open this in Google Colab (Runtime → Change runtime type → GPU is nice but not required), then run the cells top to bottom.

**What success looks like:** at the end you should see a clearly **positive** mean logit difference (roughly +2 to +4) and an accuracy near 100%. That confirms GPT-2 solves IOI, so the baseline is sound and we can build the comparison on top of it.

## 1. Install and load the model
TransformerLens wraps GPT-2 so we can read and edit its internals later. The first install takes a minute or two.

In [ ]:
!pip -q install transformer_lens

In [ ]:
import torch, itertools, random
import numpy as np
from transformer_lens import HookedTransformer

torch.set_grad_enabled(False)  # we are only running the model, not training it
device = "cuda" if torch.cuda.is_available() else "cpu"

# 'gpt2' is the 124M-parameter 'small' model
model = HookedTransformer.from_pretrained("gpt2", device=device)
print(f"Loaded GPT-2 small on {device}: {model.cfg.n_layers} layers x {model.cfg.n_heads} heads = {model.cfg.n_layers*model.cfg.n_heads} attention heads")

## 2. Choose names and templates
IOI sentences mention two people (the *subject* S and the *indirect object* IO) and expect the model to complete with the IO — the name that has **not** just acted. We keep only names that are a single token, so the metric is clean.

In [ ]:
candidate_names = [" Mary", " John", " Tom", " James", " Anna", " Kate",
                   " Mark", " Paul", " Alice", " Sarah", " David", " Emma"]

def is_single_token(name):
    return model.to_tokens(name, prepend_bos=False).shape[1] == 1

names = [n for n in candidate_names if is_single_token(n)]
print("Single-token names in use:", names)

# In each template, {A} is the indirect object (correct answer) and {B} is the subject.
templates = [
    "When{A} and{B} went to the shop,{B} gave a drink to",
    "When{A} and{B} arrived at the party,{B} handed the keys to",
    "After{A} and{B} finished work,{B} gave the report to",
]

## 3. The logit-difference metric
For each sentence we look at the model's prediction for the next token and compute:

`logit(correct name) − logit(incorrect name)`

A **positive** value means the model prefers the correct indirect object. This continuous metric is standard for IOI and is what every later experiment will use.

In [ ]:
def logit_diff(prompt, io_name, s_name):
    tokens = model.to_tokens(prompt)          # adds a beginning-of-sequence token automatically
    logits = model(tokens)                     # shape [1, sequence, vocab]
    final_logits = logits[0, -1]               # prediction for the next token
    io_tok = model.to_single_token(io_name)
    s_tok = model.to_single_token(s_name)
    return (final_logits[io_tok] - final_logits[s_tok]).item()

## 4. Run the baseline
We build a few hundred sentences by pairing names across all templates and report the average.

In [ ]:
random.seed(0)
pairs = [(a, b) for a, b in itertools.permutations(names, 2)]
random.shuffle(pairs)
pairs = pairs[:60]   # 60 name pairs x 3 templates = 180 sentences

scores = []
for a, b in pairs:
    for t in templates:
        prompt = t.format(A=a, B=b)
        scores.append(logit_diff(prompt, io_name=a, s_name=b))

scores = np.array(scores)
print(f"Sentences evaluated:            {len(scores)}")
print(f"Mean logit difference:          {scores.mean():.3f}")
print(f"Accuracy (prefers correct IO):  {(scores > 0).mean()*100:.1f}%")

# Sanity check: the baseline should clearly work.
assert scores.mean() > 1.0, "Unexpectedly low logit difference — check names/templates."
print("\nBaseline confirmed: GPT-2 small solves IOI.")

## 5. Look at a few concrete examples
It helps to see the model actually complete the sentences.

In [ ]:
for a, b in pairs[:5]:
    prompt = templates[0].format(A=a, B=b)
    ld = logit_diff(prompt, a, b)
    top_pred = model.to_string(model(model.to_tokens(prompt))[0, -1].argmax())
    print(f"{prompt!r}\n   top prediction: {top_pred!r} | correct: {a!r} | logit diff: {ld:+.2f}\n")

## What you've established
If the mean logit difference is comfortably positive and accuracy is near 100%, GPT-2 genuinely performs IOI and our metric captures it. That is the platform for the next stages:

- **Stage 2:** activation-patching baseline + an ablation harness (switching components off).
- **Stage 3:** the attribution / information-flow importances (the LM Transparency Tool method).
- **Stage 4+:** agreement and faithfulness experiments.

Save your results and commit this notebook to your project repository.